# 🎥 Тестирование и валидация моделей детекции техники на видео
Этот ноутбук предназначен для удобного тестирования любых YOLO-моделей на проектном видео (`data/IMG_2364.MOV` или любом другом видеофайле).

### Возможности:
1. 🔄 **Быстрая смена модели** — укажите путь к файлу весов (`.pt`).
2. 🎛️ **Гибкая настройка параметров** — порог уверенности (`conf`), IOU, частота кадров (`fps`), лимит кадров.
3. 🔍 **Поддержка SAHI / Tiled Slicing** — умная нарезка 4K-кадра на тайлы с перекрытием для обнаружения мелкой и удаленной техники.
4. 🖼️ **Интерактивная проверка одиночного кадра** — детальный визуальный просмотр детекций и сравнение обычного инференса с тайловым (Sliced).
5. 🎬 **Пакетная обработка видео** — покадровый анализ с прогресс-баром `tqdm` и сохранение размеченного видео.
6. 📊 **Аналитика и временная шкала** — графики присутствия техники на площадке во времени и сводная статистика.
7. ▶️ **Встроенный видеоплеер** — просмотр полученного видео прямо в ноутбуке.


In [ ]:
import os
import cv2
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO
from IPython.display import display, HTML, clear_output

# Графика matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU mode.")


In [ ]:
# ==============================================================================
# ПАРАМЕТРЫ ТЕСТИРОВАНИЯ (Настройте под свои задачи)
# ==============================================================================

# 1. Путь к модели (.pt)
# Примеры:
# - '../backend/weights/heavy_equipment_weights.pt' (новые веса тяжелой строительной техники)
# - '../backend/weights/construction_machinery.pt' (веса строительной техники 17 классов)
# - '../backend/yolov8n.pt' (базовая модель COCO)
MODEL_PATH = '../backend/weights/heavy_equipment_weights.pt'

# 2. Путь к тестовому видео
VIDEO_PATH = '../data/IMG_2364.MOV'

# 3. Порог уверенности детекции (0.0 - 1.0)
# Рекомендуется 0.5 - 0.6 для отсечения ложных срабатываний
CONF_THRESHOLD = 0.50

# 4. Порог Non-Maximum Suppression (NMS IOU)
IOU_THRESHOLD = 0.45

# 5. Механизм нарезки кадров (Tiled Slicing / SAHI)
# True: делит 4K кадр на перекрывающиеся тайлы + общий кадр (лучше находит мелкую технику)
# False: стандартный инференс YOLO на всем кадре целиком
USE_SLICING = True

# 6. Частота выборки кадров (кадров в секунду)
# Например, 1.0 = обрабатывать 1 кадр в секунду видео (быстро и наглядно)
# None = обрабатывать каждый кадр исходного видео
SAMPLE_FPS = 1.0

# 7. Ограничение количества обрабатываемых кадров для быстрого теста
# Например, 30-40 кадров для быстрой проверки за 20-40 секунд.
# Установите None для обработки всего видео.
MAX_FRAMES = 40

# 8. Путь для сохранения результирующего видео
OUTPUT_VIDEO_PATH = 'output_test_run.mp4'

# Проверка путей
assert os.path.exists(MODEL_PATH), f"Файл модели не найден: {MODEL_PATH}"
assert os.path.exists(VIDEO_PATH), f"Файл видео не найден: {VIDEO_PATH}"
print(f"✓ Модель: {os.path.abspath(MODEL_PATH)}")
print(f"✓ Видео:  {os.path.abspath(VIDEO_PATH)}")


In [ ]:
# Загрузка модели YOLO
print(f"Загрузка модели из {MODEL_PATH}...")
model = YOLO(MODEL_PATH)

# Информация о классах
class_names = model.names
print(f"\n✓ Модель успешно загружена!")
print(f"Количество классов: {len(class_names)}")
print("Список поддерживаемых классов:")
for cls_id, cls_name in class_names.items():
    print(f"  [{cls_id}] -> {cls_name}")


In [ ]:
# Получение метаданных видео
cap = cv2.VideoCapture(VIDEO_PATH)
orig_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = total_frames / orig_fps if orig_fps > 0 else 0
cap.release()

print("Метаданные исходного видео:")
print(f" - Разрешение: {width} x {height}")
print(f" - Частота кадров (FPS): {orig_fps:.2f}")
print(f" - Всего кадров: {total_frames}")
print(f" - Длительность: {duration_sec:.1f} сек ({duration_sec / 60:.2f} мин)")


In [ ]:
# Цветовая палитра для классов (стабильные различимые цвета)
np.random.seed(42)
COLOR_PALETTE = {}
for cls_id in class_names.keys():
    # Генерируем сочные цвета
    color = tuple(int(c) for c in np.random.randint(50, 230, size=3))
    COLOR_PALETTE[cls_id] = color

def nms_boxes(boxes, scores, iou_threshold=0.45):
    """Векторный Non-Maximum Suppression (NMS)"""
    if len(boxes) == 0:
        return []
    boxes = np.array(boxes, dtype=np.float32)
    scores = np.array(scores, dtype=np.float32)
    
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h
        ovr = inter / (areas[i] + areas[order[1:]] - inter + 1e-6)
        
        inds = np.where(ovr <= iou_threshold)[0]
        order = order[inds + 1]
    return keep

def detect_sliced(frame_rgb, model, conf_thresh=0.5, iou_thresh=0.45):
    """
    Tiled Slicing инференс для 4K кадров:
    Делит кадр на сетку 2x2 с перекрытием 15% + полный кадр, объединяя детекции через NMS.
    """
    h, w, _ = frame_rgb.shape
    crops = []
    crop_meta = []
    
    # 1. Сетка 2x2 с перекрытием
    overlap = 0.15
    tile_w = int(w * (0.5 + overlap / 2))
    tile_h = int(h * (0.5 + overlap / 2))
    
    x_starts = [0, w - tile_w]
    y_starts = [0, h - tile_h]
    
    for y1 in y_starts:
        for x1 in x_starts:
            x2 = min(x1 + tile_w, w)
            y2 = min(y1 + tile_h, h)
            crop = frame_rgb[y1:y2, x1:x2]
            crops.append(crop)
            crop_meta.append((x1, y1))
            
    # 2. Добавляем полнокадровый ресайз
    crops.append(frame_rgb)
    crop_meta.append((0, 0))
    
    # Пакетный инференс (быстро)
    results = model(crops, conf=conf_thresh, iou=iou_thresh, verbose=False)
    
    all_boxes = []
    all_scores = []
    all_cls_ids = []
    
    for idx, (res, (ox, oy)) in enumerate(zip(results, crop_meta)):
        boxes = res.boxes
        if boxes is None or len(boxes) == 0:
            continue
        for box in boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            score = float(box.conf[0].cpu().numpy())
            cls_id = int(box.cls[0].cpu().numpy())
            
            # Смещаем координаты тайла обратно в координаты кадра
            gx1 = xyxy[0] + ox
            gy1 = xyxy[1] + oy
            gx2 = xyxy[2] + ox
            gy2 = xyxy[3] + oy
            
            all_boxes.append([gx1, gy1, gx2, gy2])
            all_scores.append(score)
            all_cls_ids.append(cls_id)
            
    if not all_boxes:
        return []
        
    # NMS по каждому классу раздельно
    final_detections = []
    unique_classes = set(all_cls_ids)
    for c in unique_classes:
        c_indices = [i for i, cid in enumerate(all_cls_ids) if cid == c]
        c_boxes = [all_boxes[i] for i in c_indices]
        c_scores = [all_scores[i] for i in c_indices]
        
        keep_idx = nms_boxes(c_boxes, c_scores, iou_threshold=iou_thresh)
        for ki in keep_idx:
            orig_i = c_indices[ki]
            final_detections.append({
                'box': all_boxes[orig_i],
                'conf': all_scores[orig_i],
                'class_id': c,
                'class_name': class_names.get(c, f"class_{c}")
            })
            
    return final_detections

def detect_standard(frame_rgb, model, conf_thresh=0.5, iou_thresh=0.45):
    """Стандартный инференс YOLO на кадре целиком"""
    results = model(frame_rgb, conf=conf_thresh, iou=iou_thresh, verbose=False)
    detections = []
    for res in results:
        boxes = res.boxes
        if boxes is None:
            continue
        for box in boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            score = float(box.conf[0].cpu().numpy())
            cls_id = int(box.cls[0].cpu().numpy())
            detections.append({
                'box': [float(xyxy[0]), float(xyxy[1]), float(xyxy[2]), float(xyxy[3])],
                'conf': score,
                'class_id': cls_id,
                'class_name': class_names.get(cls_id, f"class_{cls_id}")
            })
    return detections

def draw_detections(frame, detections):
    """Отрисовка рамок и подписей"""
    img = frame.copy()
    h_img, w_img = img.shape[:2]
    # Масштаб шрифта зависит от разрешения
    font_scale = max(0.5, w_img / 2000.0)
    thickness = max(2, int(w_img / 1000.0))
    
    for det in detections:
        box = [int(v) for v in det['box']]
        cls_id = det['class_id']
        cls_name = det['class_name']
        conf = det['conf']
        color = COLOR_PALETTE.get(cls_id, (0, 255, 0))
        
        # Bounding Box
        cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), color, thickness)
        
        # Label с фоном
        label = f"{cls_name} {conf:.0%}"
        (lbl_w, lbl_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
        y_text = max(box[1] - 5, lbl_h + 5)
        
        cv2.rectangle(img, (box[0], y_text - lbl_h - 4), (box[0] + lbl_w + 6, y_text + baseline - 2), color, -1)
        cv2.putText(img, label, (box[0] + 3, y_text - 2), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness=max(1, thickness-1), lineType=cv2.LINE_AA)
        
    return img


In [ ]:
# ==============================================================================
# 1. ТЕСТИРОВАНИЕ НА ОДИНОЧНОМ КАДРЕ (Сравнение режимов)
# ==============================================================================

# Возьмем кадр из середины видео (например, на 10-й секунде)
target_sec = 10.0
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_MSEC, target_sec * 1000.0)
ret, test_frame_bgr = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Не удалось прочитать тестовый кадр из видео.")

test_frame_rgb = cv2.cvtColor(test_frame_bgr, cv2.COLOR_BGR2RGB)

print(f"Тестовый кадр на {target_sec} сек извлечен. Размер: {test_frame_rgb.shape[1]}x{test_frame_rgb.shape[0]}")

# 1. Стандартный инференс YOLO
t0 = time.time()
dets_std = detect_standard(test_frame_rgb, model, conf_thresh=CONF_THRESHOLD, iou_thresh=IOU_THRESHOLD)
std_time = time.time() - t0

# 2. Инференс с нарезкой кадра (Sliced / SAHI)
t0 = time.time()
dets_sliced = detect_sliced(test_frame_rgb, model, conf_thresh=CONF_THRESHOLD, iou_thresh=IOU_THRESHOLD)
sliced_time = time.time() - t0

print(f"Стандартный YOLO: {len(dets_std)} объектов обнаружено ({std_time:.3f} сек)")
print(f"Sliced (Тайловый): {len(dets_sliced)} объектов обнаружено ({sliced_time:.3f} сек)")

# Отрисовка
img_std = draw_detections(test_frame_rgb, dets_std)
img_sliced = draw_detections(test_frame_rgb, dets_sliced)

# Отображение сравнения
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(img_std)
axes[0].set_title(f"Стандартный YOLO ({len(dets_std)} об.) - {std_time:.2f}s", fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_sliced)
axes[1].set_title(f"Sliced YOLO (2x2 Grid) ({len(dets_sliced)} об.) - {sliced_time:.2f}s", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Таблица обнаруженных объектов
print("\nДетализация обнаруженных объектов (Sliced режим):")
df_frame = pd.DataFrame([
    {
        'Класс': d['class_name'],
        'Уверенность': f"{d['conf']:.1%}",
        'X1': int(d['box'][0]),
        'Y1': int(d['box'][1]),
        'X2': int(d['box'][2]),
        'Y2': int(d['box'][3]),
        'Ширина (px)': int(d['box'][2] - d['box'][0]),
        'Высота (px)': int(d['box'][3] - d['box'][1]),
    }
    for d in dets_sliced
])
if not df_frame.empty:
    display(df_frame)
else:
    print("Объектов с заданной уверенностью не обнаружено.")


In [ ]:
# ==============================================================================
# 2. ПАКЕТНАЯ ОБРАБОТКА ВСЕГО ВИДЕО С СОХРАНЕНИЕМ РЕЗУЛЬТАТА
# ==============================================================================

cap = cv2.VideoCapture(VIDEO_PATH)
in_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_in_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Расчет шага выборки кадров
step = 1
out_fps = in_fps
if SAMPLE_FPS is not None and SAMPLE_FPS > 0:
    step = max(1, int(round(in_fps / SAMPLE_FPS)))
    out_fps = SAMPLE_FPS

frames_to_process = total_in_frames // step
if MAX_FRAMES is not None:
    frames_to_process = min(frames_to_process, MAX_FRAMES)

print(f"Старт обработки видео:")
print(f" - Входной файл: {VIDEO_PATH}")
print(f" - Выходной файл: {OUTPUT_VIDEO_PATH}")
print(f" - Шаг кадров: каждые {step} кадров ({out_fps:.1f} FPS в результате)")
print(f" - Кадров к обработке: {frames_to_process}")
print(f" - Режим Slicing: {'ВКЛЮЧЕН (2x2 + Full)' if USE_SLICING else 'ВЫКЛЮЧЕН (Full Frame)'}")
print(f" - Порог уверенности: {CONF_THRESHOLD:.0%}\n")

# Настройка VideoWriter (mp4v кодек для широкой совместимости)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, out_fps, (frame_w, frame_h))

telemetry_records = []
current_frame_idx = 0
processed_count = 0

t_start = time.time()
pbar = tqdm(total=frames_to_process, desc="Обработка видео", unit="кадр")

while cap.isOpened() and processed_count < frames_to_process:
    ret, frame_bgr = cap.read()
    if not ret:
        break
        
    if current_frame_idx % step == 0:
        sec = current_frame_idx / in_fps
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        
        # Детекция
        if USE_SLICING:
            dets = detect_sliced(frame_rgb, model, conf_thresh=CONF_THRESHOLD, iou_thresh=IOU_THRESHOLD)
        else:
            dets = detect_standard(frame_rgb, model, conf_thresh=CONF_THRESHOLD, iou_thresh=IOU_THRESHOLD)
            
        # Запись телеметрии
        counts_by_class = {}
        for d in dets:
            cname = d['class_name']
            counts_by_class[cname] = counts_by_class.get(cname, 0) + 1
            telemetry_records.append({
                'frame_idx': current_frame_idx,
                'timestamp_sec': round(sec, 2),
                'class_name': cname,
                'confidence': d['conf'],
                'box_w': d['box'][2] - d['box'][0],
                'box_h': d['box'][3] - d['box'][1]
            })
            
        # Отрисовка
        annotated_rgb = draw_detections(frame_rgb, dets)
        
        # Информационная плашка сверху кадра
        counts_str = ", ".join([f"{k}: {v}" for k, v in counts_by_class.items()]) if counts_by_class else "Нет объектов"
        overlay_text = f"Time: {sec:.1f}s | Objs: {len(dets)} [{counts_str}]"
        annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
        cv2.putText(annotated_bgr, overlay_text, (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 
                    max(0.7, frame_w / 2500.0), (0, 255, 255), 2, cv2.LINE_AA)
        
        out_writer.write(annotated_bgr)
        processed_count += 1
        pbar.update(1)
        
    current_frame_idx += 1

cap.release()
out_writer.release()
pbar.close()

total_duration = time.time() - t_start
fps_actual = processed_count / total_duration if total_duration > 0 else 0
print(f"\n✓ Видео успешно сохранено в: {OUTPUT_VIDEO_PATH}")
print(f"✓ Обработано {processed_count} кадров за {total_duration:.1f}с (скорость: {fps_actual:.2f} FPS)")


In [ ]:
# ==============================================================================
# 3. АНАЛИТИКА: ГРАФИКИ ПРИСУТСТВИЯ И РАСПРЕДЕЛЕНИЕ ТЕХНИКИ
# ==============================================================================

df_telemetry = pd.DataFrame(telemetry_records)

if df_telemetry.empty:
    print("За время обработки видео объекты не были зафиксированы.")
else:
    print(f"Всего зафиксировано детекций: {len(df_telemetry)}")
    
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # 1. График динамики присутствия классов по времени
    pivot_df = df_telemetry.groupby(['timestamp_sec', 'class_name']).size().unstack(fill_value=0)
    pivot_df.plot(ax=axes[0], marker='o', linewidth=2, alpha=0.85)
    axes[0].set_title("Динамика обнаружения техники во времени (посекундно)", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Время на видео (секунды)")
    axes[0].set_ylabel("Количество единиц техники")
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(title="Класс техники", bbox_to_anchor=(1.02, 1), loc='upper left')
    
    # 2. Средняя уверенность модели по каждому классу
    conf_by_class = df_telemetry.groupby('class_name')['confidence'].agg(['count', 'mean', 'min', 'max']).reset_index()
    conf_by_class['mean'] = conf_by_class['mean'] * 100
    sns.barplot(data=conf_by_class, x='class_name', y='mean', ax=axes[1], palette='Blues_d')
    axes[1].set_title("Средняя уверенность модели (Confidence %) по классам", fontsize=14, fontweight='bold')
    axes[1].set_xlabel("Класс техники")
    axes[1].set_ylabel("Средняя уверенность (%)")
    axes[1].set_ylim(0, 100)
    for p in axes[1].patches:
        height = p.get_height()
        if not np.isnan(height):
            axes[1].annotate(f"{height:.1f}%", (p.get_x() + p.get_width() / 2., height / 2),
                             ha='center', va='center', color='white', fontweight='bold', fontsize=12)
    axes[1].grid(axis='y', linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()
    
    # Сводная таблица
    print("\nСводная статистика работы модели:")
    summary_df = df_telemetry.groupby('class_name').agg(
        Количество_детекций=('confidence', 'count'),
        Средняя_уверенность=('confidence', lambda x: f"{x.mean():.1%}"),
        Мин_уверенность=('confidence', lambda x: f"{x.min():.1%}"),
        Макс_уверенность=('confidence', lambda x: f"{x.max():.1%}"),
        Средний_размер_px=('box_w', lambda x: f"{int(x.mean())}x{int(df_telemetry.loc[x.index, 'box_h'].mean())}")
    ).reset_index()
    display(summary_df)


In [ ]:
# ==============================================================================
# 4. ПРОСМОТР РЕЗУЛЬТИРУЮЩЕГО ВИДЕО В НОУТБУКЕ
# ==============================================================================

import io
import base64

if os.path.exists(OUTPUT_VIDEO_PATH):
    file_size_mb = os.path.getsize(OUTPUT_VIDEO_PATH) / (1024 * 1024)
    print(f"Размер файла {OUTPUT_VIDEO_PATH}: {file_size_mb:.2f} МБ")
    
    # Если файл до 50 МБ, можем встроить его через data URI
    if file_size_mb < 50:
        video_bytes = open(OUTPUT_VIDEO_PATH, "rb").read()
        video_b64 = base64.b64encode(video_bytes).decode()
        video_html = f"""
        <div style="text-align: center; margin: 20px 0;">
            <h3>Результат детекции техники на видео:</h3>
            <video width="850" height="480" controls style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
                <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
                Ваш браузер не поддерживает встроенное воспроизведение video.
            </video>
        </div>
        """
        display(HTML(video_html))
    else:
        print(f"Файл видео слишком большой для base64 ({file_size_mb:.1f} МБ).")
        print(f"Вы можете открыть файл напрямую в медиаплеере по пути: {os.path.abspath(OUTPUT_VIDEO_PATH)}")
else:
    print("Файл видео еще не создан.")
